# 01 · Build the scenario

**Goal:** turn a list of real Bangladeshi addresses into a table of coordinates and parcel weights, then see it on a map.


In [1]:
!pip install -q ortools folium geopy requests pandas numpy matplotlib

### Imports and configuration
- `SEED = 42` makes the random parcel weights reproducible.
- `CITY` comes from the `EMROUTE_CITY` environment variable (default `dhaka`), so the same notebooks run for both cities.
- `CITY_BOX` is a bounding box (south-west corner, north-east corner). Geocoding is restricted to it so a same-named place elsewhere in the country can't sneak in.

In [16]:
import math
import os

import folium
import numpy as np
import pandas as pd
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

SEED = 42
CITY = os.environ.get("EMROUTE_CITY", "dhaka")   # "dhaka" or "sylhet"

DATA_DIR = "../data"
OUTPUT_DIR = "../outputs"

# Geocoding search area: (south-west corner), (north-east corner).
CITY_BOX = {
    "dhaka":  [(23.65, 90.30), (23.95, 90.55)],
    "sylhet": [(24.80, 91.75), (25.00, 92.00)],
}[CITY]

### Addresses for both cities
- **Dhaka**: flat and congested, so the load and speed effects should dominate.
- **Sylhet**: the hilliest major city, the only place the gradient effect has a chance to show.

The depot is placed at **index 0** of `all_addresses`. Every matrix and route later in the project relies on "location 0 = depot".

In [15]:
dhaka_depot = "Tejgaon Industrial Area, Dhaka, Bangladesh"

dhaka_stops = [
    "Gulshan 1, Dhaka, Bangladesh",
    "Gulshan 2, Dhaka, Bangladesh",
    "Banani, Dhaka, Bangladesh",
    "Mohakhali, Dhaka, Bangladesh",
    "Dhanmondi 27, Dhaka, Bangladesh",
    "Lalmatia, Dhaka, Bangladesh",
    "Mohammadpur, Dhaka, Bangladesh",
    "Farmgate, Dhaka, Bangladesh",
    "Kawran Bazar, Dhaka, Bangladesh",
    "Shahbagh, Dhaka, Bangladesh",
    "New Market, Dhaka, Bangladesh",
    "Azimpur, Dhaka, Bangladesh",
    "Motijheel, Dhaka, Bangladesh",
    "Paltan, Dhaka, Bangladesh",
    "Malibagh, Dhaka, Bangladesh",
    "Rampura, Dhaka, Bangladesh",
    "Badda, Dhaka, Bangladesh",
    "Bashundhara Residential Area, Dhaka, Bangladesh",
    "Mirpur 10, Dhaka, Bangladesh",
    "Uttara Sector 7, Dhaka, Bangladesh",
]

# Comments give each place's elevation (SRTM).
sylhet_depot = "Kadamtali, Sylhet, Bangladesh"                      # 13 m

sylhet_stops = [
    "Zindabazar, Sylhet, Bangladesh",                               # 21 m
    "Lamabazar, Sylhet, Bangladesh",                                # 21 m
    "Chowhatta, Sylhet, Bangladesh",                                # 24 m
    "MAG Osmani Medical College, Sylhet, Bangladesh",               # 23 m
    "Mirabazar, Sylhet, Bangladesh",                                # 18 m
    "Subid Bazar, Sylhet, Bangladesh",                              # 21 m
    "Ambarkhana, Sylhet, Bangladesh",                               # 25 m
    "Hazrat Shahjalal Mazar, Sylhet, Bangladesh",                   # 25 m
    "Shahi Eidgah, Sylhet, Bangladesh",                             # 29 m
    "Khadimnagar, Sylhet, Bangladesh",                              # 57 m  <- high point
    "Sylhet Cadet College, Sylhet, Bangladesh",                     # 26 m
    "Majortila, Sylhet, Bangladesh",                                # 24 m
    "Tilagor, Sylhet, Bangladesh",                                  # 21 m
    "Shahjalal Upashahar, Sylhet, Bangladesh",                      # 16 m
    "Kumarpara, Sylhet, Bangladesh",                                # 31 m
    "Osmani International Airport, Sylhet, Bangladesh",             # 12 m
]

if CITY == "dhaka":
    depot_address, stop_addresses = dhaka_depot, dhaka_stops
else:
    depot_address, stop_addresses = sylhet_depot, sylhet_stops

all_addresses = [depot_address] + stop_addresses   # index 0 = depot

print(f"City: {CITY}")
print(f"{len(all_addresses)} locations total (1 depot + {len(stop_addresses)} stops)")

City: dhaka
21 locations total (1 depot + 20 stops)


### Geocode addresses → (lat, lon)
Nominatim (OpenStreetMap) is queried at most once every 1.1 s, per its usage policy. `bounded=True` makes an out-of-box match fail loudly instead of returning a wrong point.

Second safety net: each stop's straight-line distance to the depot must be ≤ 25 km. It is computed with the **haversine formula**:

$$a = \sin^2\!\left(\tfrac{\Delta\varphi}{2}\right) + \cos\varphi_1\cos\varphi_2\,\sin^2\!\left(\tfrac{\Delta\lambda}{2}\right), \qquad d = 2R\,\arcsin\sqrt{a}$$

where $\varphi$ = latitude, $\lambda$ = longitude (in radians) and $R = 6371$ km is the Earth's radius.

In [17]:
geolocator = Nominatim(
    user_agent="emroute-last-mile/0.1 (github.com/TushyTasnim/last-mile-delivery)"
)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)   # Nominatim allows 1 request/s

latitudes = []
longitudes = []

for address in all_addresses:
    location = geocode(address, viewbox=CITY_BOX, bounded=True)
    if location is None:
        raise ValueError(f"Could not geocode inside the {CITY} bounding box: {address!r}. "
                         "Replace it with a nearby landmark that is in OpenStreetMap.")

    latitudes.append(location.latitude)
    longitudes.append(location.longitude)
    print(f"OK  {address:<50} -> ({location.latitude:.5f}, {location.longitude:.5f})")

print()


def haversine_km(lat1, lon1, lat2, lon2):
    """Straight-line distance between two (lat, lon) points, in km."""
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2)
    return 6371 * 2 * math.asin(math.sqrt(a))


# Catch a bad match inside the box: no stop should be absurdly far from the depot.
MAX_KM_FROM_DEPOT = 25.0

for address, lat, lon in zip(all_addresses, latitudes, longitudes):
    km = haversine_km(latitudes[0], longitudes[0], lat, lon)
    if km > MAX_KM_FROM_DEPOT:
        raise ValueError(f"{address!r} geocoded {km:.1f} km from the depot. "
                         "That is almost certainly the wrong place.")

print(f"All stops within {MAX_KM_FROM_DEPOT:.0f} km of the depot.")

OK  Tejgaon Industrial Area, Dhaka, Bangladesh         -> (23.76488, 90.40580)
OK  Gulshan 1, Dhaka, Bangladesh                       -> (23.78055, 90.41690)
OK  Gulshan 2, Dhaka, Bangladesh                       -> (23.79472, 90.41370)
OK  Banani, Dhaka, Bangladesh                          -> (23.79032, 90.40770)
OK  Mohakhali, Dhaka, Bangladesh                       -> (23.77999, 90.40832)
OK  Dhanmondi 27, Dhaka, Bangladesh                    -> (23.75409, 90.37135)
OK  Lalmatia, Dhaka, Bangladesh                        -> (23.75670, 90.36916)
OK  Mohammadpur, Dhaka, Bangladesh                     -> (23.75939, 90.36633)
OK  Farmgate, Dhaka, Bangladesh                        -> (23.75904, 90.38711)
OK  Kawran Bazar, Dhaka, Bangladesh                    -> (23.75143, 90.39486)
OK  Shahbagh, Dhaka, Bangladesh                        -> (23.73946, 90.39598)
OK  New Market, Dhaka, Bangladesh                      -> (23.73302, 90.38468)
OK  Azimpur, Dhaka, Bangladesh                      

### Parcel weights
Each stop's parcel weight is drawn from a **log-normal** distribution: $\ln w \sim \mathcal{N}(\mu,\ \sigma)$ with $\mu = \ln 50$ and $\sigma = 0.55$. So the median parcel is $e^{\mu} = 50$ kg; most stops are light and a few are heavy, which is how real delivery loads look. The generator is seeded, so the weights are the same on every run.

The weights are then clipped to 5–180 kg (no empty or absurd stops) and rounded to whole kg. Finally the depot is added at the front with demand 0, so `demands[i]` lines up with location `i`.

In [9]:
rng = np.random.default_rng(SEED)

demands = rng.lognormal(mean=np.log(50), sigma=0.55, size=len(stop_addresses))
demands = np.clip(demands, 5, 180).round(0)
demands = np.concatenate([[0], demands])   # depot first, with no demand

print(f"Total demand: {demands.sum():.0f} kg")
print(f"Min / median / max per stop: "
      f"{demands[1:].min():.0f} / {np.median(demands[1:]):.0f} / {demands[1:].max():.0f} kg")

Total demand: 1085 kg
Min / median / max per stop: 17 / 53 / 93 kg


### Fleet and feasibility
3 vans, each with a 600 kg payload limit and a 1500 kg curb (empty) weight.
- A full van is $600 / 1500 = 40\%$ heavier than an empty one.
- Fleet capacity $= 3 \times 600 = 1800$ kg.
- Utilisation $= \dfrac{\text{total demand}}{\text{fleet capacity}}$.

The problem is only solvable if (1) total demand ≤ fleet capacity, and (2) no single stop is heavier than one van can carry.

In [12]:
NUM_VEHICLES = 3        # vans leaving the depot
CAPACITY_KG = 600       # max payload per van
CURB_WEIGHT_KG = 1500   # weight of the empty van

total_capacity = NUM_VEHICLES * CAPACITY_KG

print(f"Empty van: {CURB_WEIGHT_KG} kg")
print(f"Full van:  {CURB_WEIGHT_KG + CAPACITY_KG} kg "
      f"(+{100 * CAPACITY_KG / CURB_WEIGHT_KG:.0f}% mass)")
print(f"\nFleet capacity: {total_capacity} kg")
print(f"Total demand:   {demands.sum():.0f} kg")
print(f"Utilisation:    {100 * demands.sum() / total_capacity:.0f}%")

assert demands.sum() <= total_capacity, "Not enough vehicle capacity! Add a van."
assert demands[1:].max() <= CAPACITY_KG, "A single stop exceeds one van's capacity!"
print("\nFeasible.")

Empty van: 1500 kg
Full van:  2100 kg (+40% mass)

Fleet capacity: 1800 kg
Total demand:   1085 kg
Utilisation:    60%

Feasible.


### Save the scenario
One row per location (index, address, coordinates, demand, depot flag) → `data/stops_{city}.csv`.

In [13]:
stops = pd.DataFrame({
    "index":     range(len(all_addresses)),   # 0 = depot
    "address":   all_addresses,
    "lat":       latitudes,
    "lon":       longitudes,
    "demand_kg": demands,
    "is_depot":  [True] + [False] * len(stop_addresses),
})

path = f"{DATA_DIR}/stops_{CITY}.csv"
stops.to_csv(path, index=False)

print(f"Saved -> {path}")
stops.head(10)

Saved -> ../data/stops_dhaka.csv


,index,address,lat,lon,demand_kg,is_depot
0,0,"Tejgaon Industrial Area, Dhaka, Bangladesh",23.764878,90.405798,0.0,True
1,1,"Gulshan 1, Dhaka, Bangladesh",23.780553,90.416900,59.0,False
2,2,"Gulshan 2, Dhaka, Bangladesh",23.794719,90.413699,28.0,False
3,3,"Banani, Dhaka, Bangladesh",23.790321,90.407696,76.0,False
4,4,"Mohakhali, Dhaka, Bangladesh",23.779989,90.408324,84.0,False
5,5,"Dhanmondi 27, Dhaka, Bangladesh",23.754089,90.371351,17.0,False
6,6,"Lalmatia, Dhaka, Bangladesh",23.756701,90.369155,24.0,False
7,7,"Mohammadpur, Dhaka, Bangladesh",23.759389,90.366325,54.0,False
8,8,"Farmgate, Dhaka, Bangladesh",23.759039,90.387111,42.0,False
9,9,"Kawran Bazar, Dhaka, Bangladesh",23.751431,90.394860,50.0,False


### Map the stops
Depot = red home marker. Each stop is a circle with radius $3 + \text{demand}/20$ px, so heavier deliveries look bigger. Saved to `outputs/stops_{city}.html`.

In [18]:
ESRI_LIGHT_GRAY = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
                   "Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

depot = stops.iloc[0]
m = folium.Map(location=[depot["lat"], depot["lon"]], zoom_start=12,
               tiles=ESRI_LIGHT_GRAY, attr="Tiles &copy; Esri")

folium.Marker([depot["lat"], depot["lon"]], popup="DEPOT",
              icon=folium.Icon(color="red", icon="home")).add_to(m)

# One circle per stop; heavier parcels get bigger circles.
for _, row in stops.iloc[1:].iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]],
        radius=3 + row["demand_kg"] / 20,
        popup=f"{row['address']}<br>{row['demand_kg']:.0f} kg",
        color="blue",
        fill=True,
        fill_opacity=0.6,
    ).add_to(m)

m.save(f"{OUTPUT_DIR}/stops_{CITY}.html")
m

## Conclusions

| | Dhaka | Sylhet |
|---|---|---|
| Stops (+ depot) | 20 | 16 |
| Total demand | 1085 kg | 864 kg |
| Fleet utilisation (3 × 600 kg) | 60% | 48% |

- All addresses geocode inside the city box and within 25 km of the depot, so the scenario is built from real, correctly placed locations.
- Parcel weights are moderate (Dhaka: 17 / 53 / 93 kg min / median / max). No stop comes close to a van's capacity, so the problem is comfortably feasible.
- With 60% utilisation, the demand fits in **two** vans; the solver in Notebook 4 indeed leaves the third van at the depot.
- The two geocoding guards are not optional. An early run placed "Bandar Bazar, Sylhet" 45.7 km away, and that single point accounted for 66% of all driving. A wrong coordinate corrupts every later result without any error, so failing loudly is the right behaviour.